<div style="
    border: 2px solid #4CAF50; 
    padding: 15px; 
    background-color: #f4f4f4; 
    border-radius: 10px; 
    align-items: center;">

<h1 style="margin: 0; color: #4CAF50;">Neural Networks: Ein Beispiel (Regression) (Lösung)</h1>
<h2 style="margin: 5px 0; color: #555;">DSAI</h2>
<h3 style="margin: 5px 0; color: #555;">Jakob Eggl</h3>

<div style="flex-shrink: 0;">
    <img src="https://www.htl-grieskirchen.at/wp/wp-content/uploads/2022/11/logo_bildschirm-1024x503.png" alt="Logo" style="width: 250px; height: auto;"/>
</div>
<p1> © 2025/26 Jakob Eggl. Nutzung oder Verbreitung nur mit ausdrücklicher Genehmigung des Autors.</p1>
</div>
<div style="flex: 1;">
</div>   

Wir wollen ein nun ein Neuronales Netzwerk trainieren, welches uns die Qualität eines Weißweins, basierend auf einigen numerischen Daten, vorhersagt. Dazu gegeben ist das dataset `wine.csv` von der Website https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv.

### Die Daten

Sehen wir uns zuerst das Dataset genauer an.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print(device)

In [ ]:
path_to_dataset = os.path.join("..", "..", "_data", "wine.csv")

dataframe = pd.read_csv(path_to_dataset, sep=',', header=0, decimal='.')

In [ ]:
dataframe

In [ ]:
dataframe.info()

In unserem Fall sind also keine Daten fehlend. Wir können uns nun anschauen, wie viele bzw. welche Labels dass es gibt.

In [ ]:
print(sorted(dataframe['quality'].unique()))

Es gibt also die Qualitätsstufen 3, 4, 5, 6, 7, 8, 9.

Wir betrachten noch die Anzahl an verschiedenen Qualitätsstufen.

In [ ]:
print(dataframe['quality'].value_counts().sort_index())

Wir sehen, dass die Klassen sehr unterschiedlich aufgeteilt sind. Es gibt also sehr wenige sehr schlechte Weine (Qualität 3) und sehr sehr wenige extrem gute Weine (Qualität 9).

Wir entscheiden uns in diesem Fall für ein **Regressionsmodell**. Es wäre natürlich auch möglich, ein **Klassifikationsmodell** zu trainieren, jedoch wollen wir ausnutzen, dass die **Qualitätsstufen** **geordnet** sind. Also ein Wein mit Qualitätsstufe 7.8 ist zwischen den Stufen 7 und 8 und näher an 8.

## Das Dataset und die Dataloader

Kümmern wir uns nun um die Train/Test Sets und die dazugehörigen Dataloader.

Wir starten mit einer eigenen Klasse für ein Dataset.

In [ ]:
class WineQualityDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        return x, y

Nun splitten wir unser Dataset auf in Trainings und Testdaten. Wir verwenden einen 80/20 Split.

In [ ]:
X = dataframe.drop("quality", axis=1).values
y = dataframe["quality"].values

In [ ]:
dataframe

In [ ]:
X

In [ ]:
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Wir wollen die Daten auch gleich normalisieren!

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test) # warum hier das gleiche? -> weil wir davon ausgehen, die haben die gleiche Verteilung.

In [ ]:
train_dataset = WineQualityDataset(X_train, y_train)
test_dataset = WineQualityDataset(X_test, y_test)

Nun definieren wir uns die Dataloader

In [ ]:
### HYPERPARAMETER ###
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Das Modell

Wir definieren uns nun das Modell, mit dem wir die Vorhersage treffen wollen.

In [ ]:
class WineRegressor(nn.Module):
    def __init__(self):
        super(WineRegressor, self).__init__()
        self.fc1 = nn.Linear(11, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
model = WineRegressor().to(device)

## Loss Funktion und Optimizer

Wir müssen uns nun einen geeigneten Loss definieren und auch entscheiden, welchen Optimizer wir verwenden wollen inkl. der Learning Rate usw.

Nachdem wir hier eine Regression machen wollen, wählen wir als Loss den *Mean Squared Error* (MSE) Loss.

Als Optimizer wollen wir ganz klassisch Stochastic Gradient Descent verwenden mit einer Learning Rate von $0.001$.

In [ ]:
criterion = nn.MSELoss()
learning_rate = 0.001
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

## Trainingsmethode und Evaluierung

In [ ]:
### Hyperparameter ###
epochs = 100
print_every = 5

# add validation in the training loop
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.float().to(device), batch_y.float().to(device) # float wegen Datentyp, ansonsten können die Matrizen nicht multipliziert werden
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    if (epoch) % print_every == 0:
        print(f"Epoch [{epoch}/{epochs}], Loss: {epoch_loss:.4f}")


In [ ]:
model.eval()
test_loss = 0.0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.float().to(device), batch_y.float().to(device)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        test_loss += loss.item() * batch_x.size(0)

test_loss /= len(test_loader.dataset)
print(f"Test MSE: {test_loss:.4f}")

Sind wir zufrieden?

Kurzer Vergleich mit Decision Tree und mit kNN

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
#kNN and Decision Tree for the same data

kNN = KNeighborsRegressor(n_neighbors=5)
decision_tree = DecisionTreeRegressor()

In [ ]:
kNN.fit(X_train, y_train)
decision_tree.fit(X_train, y_train)

In [ ]:
# Check how good kNN and decision tree performed by calculating the MLE on the testset

y_pred_knn = kNN.predict(X_test)
y_pred_tree = decision_tree.predict(X_test)

mse_knn = mean_squared_error(y_test, y_pred_knn)
mse_tree = mean_squared_error(y_test, y_pred_tree)

In [ ]:
mse_knn

In [ ]:
mse_tree

Geht es noch besser?

In [ ]:
decision_tree = DecisionTreeRegressor(max_depth=5)

decision_tree.fit(X_train, y_train)

y_pred_tree = decision_tree.predict(X_test)

mse_tree = mean_squared_error(y_test, y_pred_tree)

print(mse_tree)

In [ ]:
best_error_knn = float("inf")
best_k_knn = 0

for k in range(1, 52):
    kNN = KNeighborsRegressor(n_neighbors=k)
    kNN.fit(X_train, y_train)
    y_pred_knn = kNN.predict(X_test)
    mse_knn = mean_squared_error(y_test, y_pred_knn)
    if mse_knn < best_error_knn:
        best_error_knn = mse_knn
        best_k_knn = k

print(f"Best k (kNN)={best_k_knn}, Best MSE (kNN)={best_error_knn}")

Somit muss bei unserem neuronalen Netz auch noch etwas möglich sein!

## Verbesserungen und Veränderungen

In [ ]:
X_test, X_valid, y_test, y_valid = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

In [ ]:
# Normalisieren wäre jetzt auch noch wichtig, aber wir splitten ja die bereits normalisierten Daten.

In [ ]:
train_dataset = WineQualityDataset(X_train, y_train)
test_dataset = WineQualityDataset(X_test, y_test)
valid_dataset = WineQualityDataset(X_valid, y_valid)

In [ ]:
### HYPERPARAMETER ###
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class NewWineRegressor(nn.Module):
    def __init__(self):
        super(NewWineRegressor, self).__init__()
        # Info: Dropout erst später
        self.fc1 = nn.Linear(11, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)
        # self.dropout1 = nn.Dropout(p=0.3)
        # self.dropout2 = nn.Dropout(p=0.3)
        # self.dropout3 = nn.Dropout(p=0.3)

    def forward(self, x):        
        # x = self.dropout1(F.relu(self.fc1(x)))
        # x = self.dropout2(F.relu(self.fc2(x)))
        # x = self.dropout3(F.relu(self.fc3(x)))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)

In [ ]:
model = NewWineRegressor().to(device)

In [ ]:
criterion = nn.MSELoss()
learning_rate = 0.001
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
### Hyperparameter ###
epochs = 1000
print_every = 10
validate_every = 10
early_stop_patience = 5  
best_val_loss = float('inf')
epoch_val_loss = best_val_loss
epochs_no_improve = 0
early_stop = False

save_path = os.path.join("..", "models", "nn_7_best_model.pth")


for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.float().to(device), batch_y.float().to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    if epoch % validate_every == 0:
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x_val, batch_y_val in valid_loader:
                batch_x_val, batch_y_val = batch_x_val.float().to(device), batch_y_val.float().to(device)
                outputs_val = model(batch_x_val)
                loss_val = criterion(outputs_val, batch_y_val)
                val_loss += loss_val.item() * batch_x_val.size(0)

        epoch_val_loss = val_loss / len(valid_loader.dataset)

        # Early stopping logic
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epochs.")
            if epochs_no_improve == early_stop_patience:
                print("Early stopping triggered!")
                early_stop = True
                break

    if epoch % print_every == 0:
        print(f"Epoch [{epoch}/{epochs}], Training Loss: {epoch_loss:.4f}, Validation Loss: {epoch_val_loss:.4f}")

    if early_stop:
        break

In [ ]:
# load best model
model.load_state_dict(torch.load(save_path))

model.eval()
test_loss = 0.0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.float().to(device), batch_y.float().to(device)
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        test_loss += loss.item() * batch_x.size(0)

test_loss /= len(test_loader.dataset)
print(f"Test MSE: {test_loss:.4f}")

Wir sehen, dass wir quasi gleich gut sind.

Oder wir geben direkt das ganze Dataset in das Modell (i.e. Batch-Size = Länge des Datasets).

In [ ]:
# Features und Ziel trennen
X = dataframe.drop("quality", axis=1).values
y = dataframe["quality"].values

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_test, X_valid, y_test, y_valid = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42
)

# Features normalisieren (sehr wichtig für NN)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test) # auch hier wichtig, dass wir die Testdaten dann immer bzgl. Trainingsparameter normalisieren. Das muss generell so passieren.
X_valid = scaler.transform(X_valid)

# In PyTorch-Tensoren umwandeln
X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device=device)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32, device=device)
X_valid_tensor  = torch.tensor(X_valid, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=device).view(-1, 1)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32, device=device).view(-1, 1)
y_valid_tensor  = torch.tensor(y_valid, dtype=torch.float32, device=device).view(-1, 1)

In [ ]:
model = NewWineRegressor().to(device)

criterion = nn.MSELoss()
learning_rate = 0.001
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

### Hyperparameter ###
epochs = 10000
print_every = 100
validate_every = 20
early_stop_patience = 5  # Number of epochs to wait for improvement
best_val_loss = float('inf')
epoch_val_loss = best_val_loss
epochs_no_improve = 0
early_stop = False

save_path = os.path.join("..", "models", "nn_7_best_model_max_batch_size.pth")

# add validation in the training loop
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    running_loss += loss.item() * X_train_tensor.size(0)

    epoch_loss = running_loss / len(X_train_tensor)

    # Validation loop
    if epoch % validate_every == 0:
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            outputs_val = model(X_valid_tensor)
            loss_val = criterion(outputs_val, y_valid_tensor)
            val_loss += loss_val.item() * X_valid_tensor.size(0)

        epoch_val_loss = val_loss / len(X_valid_tensor)

        # Early stopping logic
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epochs.")
            if epochs_no_improve == early_stop_patience:
                print("Early stopping triggered!")
                early_stop = True
                break

    if epoch % print_every == 0:
        print(f"Epoch [{epoch}/{epochs}], Training Loss: {epoch_loss:.4f}, Validation Loss: {epoch_val_loss:.4f}")

    if early_stop:
        break

In [ ]:
model.load_state_dict(torch.load(save_path))

model.eval()
test_loss = 0.0
with torch.no_grad():
    outputs = model(X_test_tensor)
    loss = criterion(outputs, y_test_tensor)
    test_loss += loss.item() * X_test_tensor.size(0)

test_loss /= len(X_test_tensor)
print(f"Test MSE: {test_loss:.4f}")

Also viel besser als vorher und quasi gleich gut wie das beste kNN Modell (aber auch viel aufwändiger als zbsp kNN).

Was wäre wenn wir noch Dropout hinzugeben? (Spoiler: Es wird vermutlich schlechter)

> **Übung:** Verbessere das Modell wenn möglich!